# Retrieval System

FAISS-based retrieval with abnormality stratification.

In [ ]:
import faiss
import torch
import pickle
import numpy as np
import pandas as pd

## Medical Image Retriever Class

In [ ]:
class MedicalImageRetriever:
    def __init__(self, 
                 index_path='faiss_index/mimic_image_index_cpu.faiss',
                 metadata_path='faiss_index/mimic_faiss_metadata.pkl'):
        """Initialize retriever with FAISS index and metadata."""
        print("Loading FAISS index...")
        self.index = faiss.read_index(index_path)
        
        print("Loading metadata...")
        with open(metadata_path, 'rb') as f:
            self.metadata = pickle.load(f)
        
        print(f"✅ Loaded index with {self.index.ntotal} vectors")
    
    def retrieve(self, query_embedding, k=10, stratify=True, abnormality_threshold=0.5):
        """
        Retrieve similar medical images.
        
        Args:
            query_embedding: Query image embedding (torch.Tensor)
            k: Number of cases to retrieve per category
            stratify: If True, separate healthy/abnormal cases
            abnormality_threshold: Threshold for abnormality classification
        
        Returns:
            dict: Retrieved cases (stratified or flat)
        """
        # Convert to numpy
        query_np = query_embedding.numpy().astype('float32')
        if query_np.ndim == 1:
            query_np = query_np.reshape(1, -1)
        
        # Search FAISS index (get more to allow filtering)
        search_k = k * 4 if stratify else k
        scores, indices = self.index.search(query_np, k=search_k)
        
        if not stratify:
            return self._format_results(indices[0][:k], scores[0][:k])
        
        # Stratify by abnormality
        abnormal_cases = []
        healthy_cases = []
        
        for idx, score in zip(indices[0], scores[0]):
            case = self.metadata[idx].copy()
            case['similarity'] = float(score)
            
            # Check abnormality score (if available in metadata)
            abnormality_score = case.get('abnormality_score', 0)
            
            if abnormality_score > abnormality_threshold:
                abnormal_cases.append(case)
            else:
                healthy_cases.append(case)
            
            # Stop when we have enough of each
            if len(abnormal_cases) >= k and len(healthy_cases) >= k:
                break
        
        return {
            'abnormal': abnormal_cases[:k],
            'healthy': healthy_cases[:k]
        }
    
    def _format_results(self, indices, scores):
        """Format results as list of dicts."""
        results = []
        for idx, score in zip(indices, scores):
            case = self.metadata[idx].copy()
            case['similarity'] = float(score)
            results.append(case)
        return results

## Initialize Retriever

In [ ]:
retriever = MedicalImageRetriever()

## Test Retrieval

In [ ]:
# Load sample embedding
data = torch.load('embeddings/mimic_image_embeddings.pt')
query_emb = data['embeddings'][0:1]

# Retrieve without stratification
results_flat = retriever.retrieve(query_emb, k=5, stratify=False)

print("Top 5 Similar Cases:")
for i, case in enumerate(results_flat, 1):
    print(f"\n{i}. Similarity: {case['similarity']:.4f}")
    print(f"   Image: {case.get('image_path', 'N/A')}")
    if 'impression' in case:
        print(f"   Impression: {case['impression'][:100]}...")

## Test Stratified Retrieval

In [ ]:
# Retrieve with stratification
results_stratified = retriever.retrieve(query_emb, k=3, stratify=True, abnormality_threshold=0.5)

print("\n=== ABNORMAL CASES ===")
for i, case in enumerate(results_stratified['abnormal'], 1):
    print(f"\n{i}. Similarity: {case['similarity']:.4f} | Abnormality: {case.get('abnormality_score', 'N/A')}")
    if 'impression' in case:
        print(f"   Impression: {case['impression'][:100]}...")

print("\n=== HEALTHY CASES ===")
for i, case in enumerate(results_stratified['healthy'], 1):
    print(f"\n{i}. Similarity: {case['similarity']:.4f} | Abnormality: {case.get('abnormality_score', 'N/A')}")
    if 'impression' in case:
        print(f"   Impression: {case['impression'][:100]}...")

## Retrieval with Text Query

In [ ]:
# If you want to retrieve based on text description
# from embeddings notebook import BiomedCLIPEmbedder

# embedder = BiomedCLIPEmbedder()
# text_query = "cardiomegaly with pleural effusion"
# text_emb = embedder.encode_text(text_query)

# results = retriever.retrieve(text_emb, k=5, stratify=False)
# print(f"Retrieved {len(results)} cases for query: '{text_query}'")